# Wokflow  **TEST** con Full Bayesiana 

### > Copio HP de mejor salida PPT
### > Agrego IPC
### > ~~imputo nulos~~ 
### > internet
### > MIN MAX
### > sin 202006
### > undersampling

## Inicializacion

In [1]:
# limpio la memoria
Sys.time()
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

[1] "2025-11-13 05:48:59 UTC"

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,658525,35.2,1454578,77.7,1121081,59.9
Vcells,1222263,9.4,8388608,64.0,1975165,15.1


In [ ]:
plocal <- list()

# 501
plocal$qcanaritos <- 5L
plocal$min_data_in_leaf <- 20L
plocal$learning_rate <- 1.0
plocal$gradient_bound <- 0.1


plocal$APO <- 5
plocal$ksemillerio <- 1


In [ ]:
PARAM <- list()
PARAM$experimento <- "apo-505_05_01"
PARAM$semilla_primigenia <- 102191

In [4]:
setwd("/content/buckets/b1/exp")
experimento_folder <- PARAM$experimento
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

## Preprocesamiento

### Generacion de la clase_ternaria

In [ ]:
Sys.time()
require( "data.table" )

# leo el dataset
dataset <- fread("~/datasets/competencia_02_crudo.csv.gz" )

# me quedo con todo menos 202107 y 202108
dataset <- dataset[ !(foto_mes %in% c(202107L, 202108L)) ]

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
  "pos" = .I,
  numero_de_cliente,
  periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 )
]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
  shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente
]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
  ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
  clase_ternaria := "BAJA+1"
]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
  & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
  clase_ternaria := "BAJA+2"
]

# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

rm(dsimple)
gc()
Sys.time()

[1] "2025-11-13 05:48:59 UTC"

Loading required package: data.table



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,767402,41.0,1454578,77.7,1454578,77.7
Vcells,722145115,5509.6,1017395652,7762.2,846000569,6454.5


[1] "2025-11-13 05:49:15 UTC"

In [6]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
201901,BAJA+1,645
201901,BAJA+2,729
201901,CONTINUA,122899
201902,BAJA+1,733
201902,BAJA+2,707
201902,CONTINUA,123961
201903,BAJA+1,708
201903,BAJA+2,751
201903,CONTINUA,124508


### Eliminacion de Features

Completar a gusto LUEGO de realizar un analisis exploratorio de datos.
<br> No necesariamente en esta Segunda Competencia conviele eliminar los mismos campos que en la Primera ...

In [7]:
# Salsa Magica para 202106
dataset[, mprestamos_personales := NULL ]
dataset[, cprestamos_personales := NULL ]

### Data Quality

Se deben reparar los atributos del dataset que para un cierto mes TODOS sus valores son cero.
<br> Relevar en forma muy minuciosa en el dataset cuales son los  <atributo,mes> que estan dañados.
<br> Algunas alternativas de solución son:
* No hacer absolutamente nada, dejar el valor 0 tal cual está, a sabiendas que es incorrecto
* Reemplazar esos valores dañados por  NA
* Interpolar cada valor dañado por el valor del mes previo y el posterior
* Calcularlo a partir de un modelo, libreria  MICE

a este codigo de Data Quality  lo debera escribir usted

### Data Drifting

### completa con null las columnas faltantes con 0

In [ ]:
# library(data.table)
# stopifnot(is.data.table(dataset))

# id_cols  <- c("numero_de_cliente","foto_mes","clase_ternaria")
# num_cols <- setdiff(names(dataset), id_cols)
# num_cols <- num_cols[vapply(dataset[, ..num_cols], is.numeric, logical(1))]

# # (mes, columna) con TODO 0 (estricto)
# zero_map <- dataset[
#   , lapply(.SD, function(v) isTRUE(all(v == 0))), 
#   by = .(foto_mes), .SDcols = num_cols
# ]

# zero_long <- melt(
#   zero_map, id.vars = "foto_mes",
#   variable.name = "columna", value.name = "all_zero"
# )[all_zero == TRUE, .(foto_mes, columna)]

# #  convertir la columna de nombres a character (evita el factor-gate)
# zero_long[, columna := as.character(columna)]

# if (nrow(zero_long) == 0L) {
#   message("No hay (mes, columna) con TODO 0 estricto. Nada para NA-izar.")
# } else {
#   setorder(zero_long, foto_mes, columna)
#   for (i in seq_len(nrow(zero_long))) {
#     m  <- zero_long$foto_mes[i]
#     cn <- zero_long$columna[i]  # <- ahora es character

#     idx <- which(dataset$foto_mes == m)

#     # preservar tipo
#     if (is.integer(dataset[[cn]])) {
#       set(dataset, i = idx, j = cn, value = NA_integer_)
#     } else {
#       set(dataset, i = idx, j = cn, value = NA_real_)
#     }
#   }

#   # log amigable
#   cat("Reemplacé por NA las columnas con TODO 0 (estricto):\n")
#   print(zero_long[, .(columnas = toString(columna)), by = foto_mes][order(foto_mes)])
# }


# ncol(dataset)
# Sys.time()

In [ ]:
### invierte los valores de internet

In [ ]:
# Corregir inversión de valores en campo 'internet' a partir de 202010

# 1) Limpieza: cualquier cosa que no sea 0/1 -> NA (en todos los meses)
dataset[, internet := fifelse(internet %in% 0:1, as.integer(internet), NA_integer_)]

# 2) Flip: solo para meses anteriores a 202010 y con valor válido
dataset[ foto_mes < 202010 & !is.na(internet), internet := 1L - internet ]

ncol(dataset)
Sys.time()

### Data Drifting

Se debe corregir el drifting natural que ocurre en loa datos, en particular los datos monetarios que se vieron fuertemente afectados por una alta inflación
<br> Posibles métodos son:
* No hacer absolutamente nada
* Ajuste de valores monetarios por indices del tipo :
   * IPC  Indice de Precios al Consumidor
   * Dolar Oficial
   * Dolar Blue
   * UVA  Unidad de Valor Adquisitivo

a este codigo de Data Drifting lo debera escribir usted

### IPC

In [ ]:
library(data.table)

# ================= IPC desde variaciones mensuales =================
# Variaciones mensuales (%) en orden 201901, 201902, ..., 202108  (32 valores)
ipc_var <- c(
  2.9, 3.8, 4.7, 3.4, 3.1, 2.7, 2.2, 4.0, 5.9, 3.3, 4.3, 3.7,
  2.3, 2.0, 3.3, 1.5, 1.5, 2.2, 1.9, 2.7, 2.8, 3.8, 3.2, 4.0,
  4.0, 3.6, 4.8, 4.1, 3.3, 3.2, 3.0, 2.5
)

# Genero la secuencia de foto_mes 201901..202108
seq_fm <- function(from_fm, to_fm){
  dseq <- seq(as.Date(paste0(from_fm,"01"), "%Y%m%d"),
              as.Date(paste0(to_fm,"01"), "%Y%m%d"), by="month")
  as.integer(format(dseq, "%Y%m"))
}
fm_seq <- seq_fm(201901L, 202108L)

stopifnot(length(ipc_var) == length(fm_seq))

ipc <- data.table(
  foto_mes = fm_seq,
  variacion_mensual = ipc_var
)

# Índice acumulado (base libre). Pongo 100 en el primer mes y acumulo.
ipc[, indice := 100 * cumprod(1 + variacion_mensual/100)]

# ================= Deflactar a una base elegida =================
base_mes <- 202108L            # sugerido: el último mes disponible
stopifnot(base_mes %in% ipc$foto_mes)

# Traigo el índice al dataset y calculo factor relativo a la base
setkey(ipc, foto_mes)
stopifnot("foto_mes" %in% names(dataset))
dataset <- ipc[dataset, on="foto_mes"]

indice_base <- ipc[foto_mes == base_mes, indice][1]
dataset[, defl_factor := indice / indice_base]  # >1 = precios más altos que la base

# Detecto columnas de montos (m* y Master_m*/Visa_m*)
montos_m_prefix <- grep("^m", names(dataset), value = TRUE)
montos_card     <- grep("^(Master|Visa)_m", names(dataset), value = TRUE)
montos_all <- unique(c(montos_m_prefix, montos_card))
montos_all <- montos_all[vapply(dataset[, ..montos_all], is.numeric, logical(1))]

# Deflactar IN-PLACE a pesos de 'base_mes'
for (cn in montos_all) {
  dataset[, (cn) := get(cn) / pmax(defl_factor, 1e-12)]
}

# (opcional) trazabilidad y limpieza
dataset[, ipc_base := base_mes]
dataset[, c("variacion_mensual","indice","defl_factor") := NULL]

cat(sprintf("IPC aplicado (base %d). Columnas deflactadas: %d\n", base_mes, length(montos_all)))

ncol(dataset)
Sys.time()

### Feature Engineering Intra-Mes

Crear variables nuevas a partir de las existentes dentro del mismo registro, **sin** ir a buscar información histórica.
<br> El siguiente código es un mínimo ejemplo, agregar nuevos features a gusto

In [8]:
# el mes 1,2, ..12 , podria servir para detectar estacionalidad
dataset[, kmes := foto_mes %% 100]

# creo un ctr_quarter que tenga en cuenta cuando
# los clientes hace 3 menos meses que estan
# ya que seria injusto considerar las transacciones medidas en menor tiempo
dataset[, ctrx_quarter_normalizado := as.numeric(ctrx_quarter) ]
dataset[cliente_antiguedad == 1, ctrx_quarter_normalizado := ctrx_quarter * 5.0]
dataset[cliente_antiguedad == 2, ctrx_quarter_normalizado := ctrx_quarter * 2.0]
dataset[cliente_antiguedad == 3, ctrx_quarter_normalizado := ctrx_quarter * 1.2]

# variable extraida de una tesis de maestria de Irlanda, se perdió el link
dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

Sys.time()

[1] "2025-11-13 05:49:15 UTC"

### Feature Engineering Historico

In [9]:
if( !require("Rcpp")) install.packages("Rcpp", repos = "http://cran.us.r-project.org")
require("Rcpp")

Loading required package: Rcpp



In [10]:
# se calculan para los 6 meses previos el minimo, maximo y
#  tendencia calculada con cuadrados minimos
# la formula de calculo de la tendencia puede verse en
#  https://stats.libretexts.org/Bookshelves/Introductory_Statistics/Book%3A_Introductory_Statistics_(Shafer_and_Zhang)/10%3A_Correlation_and_Regression/10.04%3A_The_Least_Squares_Regression_Line
# para la maxíma velocidad esta funcion esta escrita en lenguaje C,
# y no en la porqueria de R o Python

cppFunction("NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde )
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) )
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      {
        xsum  += x[h] ;
        ysum  += y[h] ;
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ;
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}")

In [11]:
# calcula la tendencia de las variables cols de los ultimos 6 meses
# la tendencia es la pendiente de la recta que ajusta por cuadrados minimos
# La funcionalidad de ratioavg es autoria de  Daiana Sparta,  UAustral  2021

TendenciaYmuchomas <- function(
    dataset, cols, ventana = 6, tendencia = TRUE,
    minimo = TRUE, maximo = TRUE, promedio = TRUE,
    ratioavg = FALSE, ratiomax = FALSE) {
  gc(verbose= FALSE)
  # Esta es la cantidad de meses que utilizo para la historia
  ventana_regresion <- ventana

  last <- nrow(dataset)

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids <- dataset[ , numero_de_cliente ]

  vector_desde <- seq(
    -ventana_regresion + 2,
    nrow(dataset) - ventana_regresion + 1
  )

  vector_desde[1:ventana_regresion] <- 1

  for (i in 2:last) {
    if (vector_ids[i - 1] != vector_ids[i]) {
      vector_desde[i] <- i
    }
  }
  for (i in 2:last) {
    if (vector_desde[i] < vector_desde[i - 1]) {
      vector_desde[i] <- vector_desde[i - 1]
    }
  }

  for (campo in cols) {
    nueva_col <- fhistC(dataset[, get(campo)], vector_desde)

    if (tendencia) {
      dataset[, paste0(campo, "_tend", ventana) :=
        nueva_col[(0 * last + 1):(1 * last)]]
    }

    if (minimo) {
      dataset[, paste0(campo, "_min", ventana) :=
        nueva_col[(1 * last + 1):(2 * last)]]
    }

    if (maximo) {
      dataset[, paste0(campo, "_max", ventana) :=
        nueva_col[(2 * last + 1):(3 * last)]]
    }

    if (promedio) {
      dataset[, paste0(campo, "_avg", ventana) :=
        nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratioavg) {
      dataset[, paste0(campo, "_ratioavg", ventana) :=
        get(campo) / nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratiomax) {
      dataset[, paste0(campo, "_ratiomax", ventana) :=
        get(campo) / nueva_col[(2 * last + 1):(3 * last)]]
    }
  }
}

In [12]:
# Feature Engineering Historico
# Creacion de LAGs
setorder(dataset, numero_de_cliente, foto_mes)

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
  colnames(dataset),
  c("numero_de_cliente", "foto_mes", "clase_ternaria")
))

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
  paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
  by= numero_de_cliente,
  .SDcols= cols_lagueables
]

# lags de orden 2
dataset[,
  paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
  by= numero_de_cliente,
  .SDcols= cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

Sys.time()

[1] "2025-11-13 05:50:19 UTC"

In [ ]:
# parametros de Feature Engineering Historico de Tendencias
PARAM$FE_hist$Tendencias$run <- TRUE
PARAM$FE_hist$Tendencias$ventana <- 6
PARAM$FE_hist$Tendencias$tendencia <- TRUE
PARAM$FE_hist$Tendencias$minimo <- TRUE
PARAM$FE_hist$Tendencias$maximo <- TRUE
PARAM$FE_hist$Tendencias$promedio <- FALSE
PARAM$FE_hist$Tendencias$ratioavg <- FALSE
PARAM$FE_hist$Tendencias$ratiomax <- FALSE

In [14]:
# aqui se agregan las tendencias de los ultimos 6 meses

cols_lagueables <- intersect(cols_lagueables, colnames(dataset))
setorder(dataset, numero_de_cliente, foto_mes)

if( PARAM$FE_hist$Tendencias$run) {
    TendenciaYmuchomas(dataset,
    cols = cols_lagueables,
    ventana = PARAM$FE_hist$Tendencias$ventana, # 6 meses de historia
    tendencia = PARAM$FE_hist$Tendencias$tendencia,
    minimo = PARAM$FE_hist$Tendencias$minimo,
    maximo = PARAM$FE_hist$Tendencias$maximo,
    promedio = PARAM$FE_hist$Tendencias$promedio,
    ratioavg = PARAM$FE_hist$Tendencias$ratioavg,
    ratiomax = PARAM$FE_hist$Tendencias$ratiomax
  )
}

ncol(dataset)
Sys.time()

[1] 921

[1] "2025-11-13 05:51:14 UTC"

## Modelado

No hay modelado, no se hace optimizacion de hiperparametros.

## Produccion

Las decisiones que se toman para la construccion del modelo final son:
* Los positvos son  POS={"BAJA+1", "BAJA+2"}, esta es una meticulosa decisión.
* Se entrena en los treinta meses del intervalo [201901, 202104]
* Se realiza undersampling al 5%
* Se utilizan los hiperparámetros optimos encontrados en la Bayesian Optimization
   * Se escala min_data_in_leaf

### Final Training Strategy

In [ ]:
PARAM$train_final$future <- c(202106)

PARAM$train_final$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, #202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104
)

PARAM$train_final$undersampling <- 0.1  # Silvana y Sofia

In [16]:
# se filtran los meses donde se entrena el modelo final
dataset_train_final <- dataset[foto_mes %in% PARAM$train_final$training]

In [17]:
# canaritos
PARAM$train_final$lgbm$qcanaritos <- plocal$qcanaritos

cols0 <- copy(colnames(dataset_train_final))
filas <- nrow(dataset_train_final)

if( PARAM$train_final$lgbm$qcanaritos > 0 ) {
  for( i in seq(PARAM$train_final$lgbm$qcanaritos) ){
    dataset_train_final[, paste0("canarito_",i) := runif( filas) ]
  }

  # las columnas canaritos mandatoriamente van al comienzo del dataset
  cols_canaritos <- copy( setdiff( colnames(dataset_train_final), cols0 ) )
  setcolorder( dataset_train_final, c( cols_canaritos, cols0 ) )
}

Sys.time()

[1] "2025-11-13 05:51:17 UTC"

#### Registros cambio las proporciones de POS/NEG

In [18]:
# Undersampling, van todos los "BAJA+1" y "BAJA+2" y solo algunos "CONTINIA"

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train_final[, azar := runif(nrow(dataset_train_final))]
dataset_train_final[, training := 0L]

dataset_train_final[
  (azar <= PARAM$train_final$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

dataset_train_final[, azar:= NULL] # elimino la columna azar

### Target Engineering

In [19]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

dataset_train_final[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

### Final Models

Aqui SIEMPRE voy a hacer un semillerio, independientemente de si en la Bayesian Optimization calculé un semillerio en cada iteración.
<br> Entreno un LightGBM para cada semilla,  y guardo el modelo dentro de la carpeta  **modelitos**
<br> Intencionalmente en una primera etapá se generan los modelos y graban, y en una segunda etapa se leen eso modelos y se aplican a los datos del futuro

APO controla cuantas veces se repite el modelo, que se usa para promediar ganancias y reportar en la Pseudo Competencia algo razonable
<br> El modelo puede ser un LightGBM simple (ksemillerio==1)  o un Ensemble Semillerio( ksemillerio > 1 )
<br> Lamentablmente APO necesita utilizar muchas semillas, y eso demanda TIEMPO de corrida

In [20]:

PARAM$train_final$lgbm$param_completo <-  list(
  boosting= "gbdt",
  objective= "binary",
  metric= "custom",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_bin= 31L,
  min_data_in_leaf= plocal$min_data_in_leaf,  #este ya es el valor default de LightGBM

  num_iterations= 9999L, # dejo libre la cantidad de arboles, zLightGBM se detiene solo
  num_leaves= 9999L, # dejo libre la cantidad de hojas, zLightGBM sabe cuando no hacer un split
  learning_rate= plocal$learning_rate,  # se lo deja en 1.0 para que si el score esta por debajo de gradient_bound no se lo escale
    
  feature_fraction= 0.50, # un valor equilibrado, habra que probar alternativas ...
    
  canaritos= PARAM$train_final$lgbm$qcanaritos, # fundamental en zLightGBM, aqui esta el control del overfitting
  gradient_bound= plocal$gradient_bound   # default de zLightGBM
)

Sys.time()

[1] "2025-11-13 05:51:17 UTC"

In [21]:
# Semillerio Final
PARAM$train_final$APO <- plocal$APO
PARAM$train_final$ksemillerio  <- plocal$ksemillerio

PARAM$train_final$cortes <- c(8000, 8500, 9000, 9500, 10000, 10500, 11000, 11500, 12000)

In [22]:
# detach("package:lightgbm", unload= TRUE)

In [23]:
if( !require("zlightgbm") ) install.packages("https://storage.googleapis.com/open-courses/dmeyf2025-e4a2/zlightgbm_4.6.0.99.tar.gz", repos= NULL, type= "source")
require("zlightgbm")

Loading required package: zlightgbm



In [24]:
if(!require("primes")) install.packages("primes")
require("primes")

Loading required package: primes



In [25]:
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$train_final$semillas <- sample(primos)[seq( PARAM$train_final$APO*PARAM$train_final$ksemillerio )]
PARAM$train_final$semillas

[1] 974411

In [26]:
campos_buenos <- setdiff(
  colnames(dataset_train_final),
  c( "clase_ternaria", "clase01", "training", "azar")
)

In [27]:
# dejo los datos en formato LightGBM
dtrain_final <- lgb.Dataset(
  data= data.matrix(dataset_train_final[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train_final[training == 1L, clase01],
  free_raw_data= FALSE
)

cat("filas", nrow(dtrain_final), "columnas", ncol(dtrain_final), "\n")
Sys.time()

filas 235841 columnas 930 


[1] "2025-11-13 05:51:20 UTC"

In [28]:
# genero los modelitos
dir.create( "modelitos", showWarnings= FALSE)

param_completo <- copy( PARAM$train_final$lgbm$param_completo)

for( sem in PARAM$train_final$semillas ) {

  arch_modelo <- paste0("./modelitos/mod_", sem, ".txt")
  if( !file.exists( arch_modelo ) )
  {
    param_completo$seed <- sem

    modelito <- lgb.train(
      data= dtrain_final,
      param= param_completo
    )

    lgb.save( modelito, filename= arch_modelo)
    rm(modelito)
    gc()
  }
}

Sys.time()

[1] "2025-11-13 06:10:28 UTC"

### Scoring

Se hace el predict() del modelo en los datos del futuro

In [ ]:


# dfuture: solo el mes que querés predecir, sin tocar clase_ternaria
dfuture <- dataset[ foto_mes %in% PARAM$train_final$future ]

cols0 <- copy(colnames(dfuture))
filas <- nrow(dfuture)

# canaritos si los usaste en entrenamiento
if( PARAM$train_final$lgbm$qcanaritos > 0 ) {
  for( i in seq(PARAM$train_final$lgbm$qcanaritos) ){
    dfuture[, paste0("canarito_",i) := runif( filas) ]
  }
  cols_canaritos <- copy( setdiff( colnames(dfuture), cols0 ) )
  setcolorder( dfuture, c( cols_canaritos, cols0 ) )
}

cat("se agegaron canaritos")


mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

# acá voy a acumular las predicciones del ENSEMBLE global
vpred_ens <- rep(0.0, nrow(dfuture))
qmodelos  <- 0L

# carpeta para guardar un archivo por meta_modelo
dir.create("predicciones_meta_modelos", showWarnings = FALSE)

for( vapo in seq(PARAM$train_final$APO) ) {

  # predicción acumulada SOLO para este meta_modelo (puede tener varias seeds)
  vpred_mm <- rep(0.0, nrow(dfuture))
  qacum_mm <- 0L

  desde    <- 1 + (vapo - 1L) * PARAM$train_final$ksemillerio
  hasta    <- desde + PARAM$train_final$ksemillerio - 1L
  semillas <- PARAM$train_final$semillas[desde:hasta]

  for( sem in semillas ) {
    arch_modelo <- paste0("./modelitos/mod_", sem, ".txt")
    if( file.exists( arch_modelo ) ) {
      modelo_final <- lgb.load(arch_modelo)
      p <- predict(modelo_final, mfuture)

      vpred_mm <- vpred_mm + p
      qacum_mm <- qacum_mm + 1L

      rm(modelo_final)
      gc()
    }
  }

  # si este meta_modelo efectivamente predijo algo
  if( qacum_mm > 0L ) {
    # promedio interno de sus seeds
    vpred_mm <- vpred_mm / qacum_mm

    # sumo al ensemble global
    vpred_ens <- vpred_ens + vpred_mm
    qmodelos  <- qmodelos + 1L

    # guardo archivo SOLO de este meta_modelo
    tb_mm <- dfuture[, .(numero_de_cliente, foto_mes)]
    tb_mm[, meta_modelo := vapo ]
    tb_mm[, prob := vpred_mm ]

    archivo_mm <- sprintf("predicciones_meta_modelos/prediccion_meta_modelo_%02d.txt", vapo)
    fwrite(tb_mm, file = archivo_mm, sep = "\t")

    rm(tb_mm)
    gc()
  }
}

if( qmodelos == 0L ) stop("No se cargó ningún modelo, algo está mal con las semillas.")

# promedio final del ENSEMBLE entre meta_modelos
vpred_ens <- vpred_ens / qmodelos

dfuture[, prob_ens := vpred_ens ]

# ordeno por probabilidad descendente UNA sola vez
setorder(dfuture, -prob_ens)

# cortes que querés probar
cortes <- c(9000L, 9500L, 10000L, 10500L, 11000L, 11500L, 12000L)

dir.create("kaggle", showWarnings = FALSE)

for (envios in cortes) {
  # por las dudas, no romper si el corte es más grande que la cantidad de filas
  envios_efectivos <- min(envios, nrow(dfuture))

  # genero la columna Predicted para este corte
  dfuture[, Predicted := 0L]
  if (envios_efectivos > 0L) {
    dfuture[1:envios_efectivos, Predicted := 1L]
  }

  # nombre del archivo según el corte
  archivo_kaggle <- sprintf("./kaggle/KA_%s_%d.csv",
                            PARAM$experimento, envios)

  # grabo el archivo para este corte
  fwrite(
    dfuture[, .(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep  = ","
  )

  cat("Generado:", archivo_kaggle,
      "con", envios_efectivos, "envíos\n")
}

Sys.time()


In [ ]:
# # dfuture: solo el mes que querés predecir, sin tocar clase_ternaria
# dfuture <- dataset[ foto_mes %in% PARAM$train_final$future ]

# cols0 <- copy(colnames(dfuture))
# filas <- nrow(dfuture)

# # canaritos si los usaste en entrenamiento
# if( PARAM$train_final$lgbm$qcanaritos > 0 ) {
#   for( i in seq(PARAM$train_final$lgbm$qcanaritos) ){
#     dfuture[, paste0("canarito_",i) := runif( filas) ]
#   }
#   cols_canaritos <- copy( setdiff( colnames(dfuture), cols0 ) )
#   setcolorder( dfuture, c( cols_canaritos, cols0 ) )
# }

# # mismas features que usaste en training
# campos_buenos <- setdiff(
#   colnames(dfuture),
#   c("clase_ternaria")   # si estuviera, no se usa
# )

# mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

# # acá voy a acumular las predicciones de cada meta_modelo
# vpred_ens <- rep(0.0, nrow(dfuture))
# qmodelos  <- 0L

# for( vapo in seq(PARAM$train_final$APO) ) {
#   desde <- 1 + (vapo-1)*PARAM$train_final$ksemillerio
#   hasta <- desde + PARAM$train_final$ksemillerio - 1
#   semillas <- PARAM$train_final$semillas[desde:hasta]

#   for( sem in semillas ) {
#     arch_modelo <- paste0("./modelitos/mod_", sem, ".txt")
#     if( file.exists( arch_modelo ) ) {
#       modelo_final <- lgb.load(arch_modelo)
#       vpred_ens <- vpred_ens + predict(modelo_final, mfuture)
#       qmodelos  <- qmodelos + 1L
#       rm(modelo_final)
#       gc()
#     }
#   }
# }

# if( qmodelos == 0L ) stop("No se cargó ningún modelo, algo está mal con las semillas.")

# # promedio de probabilidades
# vpred_ens <- vpred_ens / qmodelos

# dfuture[, prob_ens := vpred_ens ]

# # ordeno por probabilidad descendente
# setorder(dfuture, -prob_ens)

# # corte fijo elegido en histórico (ej: 11000)
# envios <- 11000

# dfuture[, Predicted := 0L]
# dfuture[1:envios, Predicted := 1L]

# # armo archivo final
# dir.create("kaggle", showWarnings = FALSE)

# archivo_kaggle <- paste0("./kaggle/KA_", PARAM$experimento, "_", envios, ".csv")

# fwrite(
#   dfuture[, .(numero_de_cliente, Predicted)],
#   file = archivo_kaggle,
#   sep = ","
# )


In [43]:
Sys.time()

[1] "2025-11-13 06:11:43 UTC"